# 07 - Classificacao de Sentimento com GenAI (LLM)

Os notebooks 05 e 06 usaram so preco historico pra tentar prever retorno e volatilidade. Esse notebook testa uma pergunta diferente: **texto de noticia carrega informacao que preco sozinho nao carrega?**

Pra responder isso preciso de duas coisas novas:
1. Um jeito de "entender" o sentimento de uma manchete (positiva, negativa, neutra)
2. Uma forma de usar isso como feature

## Aviso importante sobre os dados

**As manchetes usadas aqui sao ficticias**, escritas por mim pra fins didaticos -- nao sao noticias reais coletadas de nenhum veiculo de imprensa. Sao baseadas em tipos de evento plausiveis pra cada empresa (resultado trimestral, variacao de preco de commodity, etc.), mas nenhuma delas aconteceu exatamente como escrita, na data escrita. Isso fica marcado explicitamente numa coluna `nota` na tabela Bronze, pra nao dar a entender que e um dataset de noticias reais.

## O que e GenAI, rapidinho

Nos notebooks 05/06 eu treinei um modelo do zero (`XGBRegressor().fit(...)`) usando meus proprios dados numericos. Um LLM (Large Language Model) e diferente: ele ja vem pre-treinado com uma quantidade gigantesca de texto, entao ele ja "sabe" reconhecer se uma frase e positiva ou negativa sem eu precisar treinar nada -- eu so preciso escrever uma boa instrucao (**prompt**) pedindo pra ele classificar. Isso se chama *zero-shot classification*.

## Uma ressalva de escopo, pra ser honesto

Esse e um dataset pequeno de proposito (24 manchetes, 2-3 por acao) -- da pra demonstrar o pipeline completo (Bronze -> Silver com LLM -> Gold), mas nao da pra tirar uma conclusao estatistica solida tipo "sentimento de noticia melhora a previsao em X%". Pra isso valer estatisticamente, precisaria de noticias reais em volume proporcional aos dias de pregao (centenas ou milhares), o que nao e viavel montar manualmente aqui. Entao o objetivo deste notebook e **demonstrar a tecnica** (classificacao de texto com GenAI, integrada ao mesmo pipeline Bronze/Silver/Gold), nao provar uma hipotese com rigor estatistico como fizemos com retorno e volatilidade.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

# Manchetes ficticias, escritas para este exercicio -- ver aviso acima.
# Cobrem as 10 acoes do pipeline, com mistura proposital de sentimento bom/ruim/neutro.
manchetes = [
    ("2022-03-10", "PETR4", "Petrobras anuncia novo recorde de producao de petroleo no pre-sal"),
    ("2022-08-15", "PETR4", "Petrobras aprova distribuicao bilionaria de dividendos aos acionistas"),
    ("2023-02-20", "PETR4", "Governo pressiona Petrobras por mudanca na politica de precos dos combustiveis"),
    ("2022-05-12", "VALE3", "Vale reporta queda no preco do minerio de ferro no mercado chines"),
    ("2023-01-18", "VALE3", "Vale anuncia investimento bilionario em expansao de mina no Para"),
    ("2023-09-05", "VALE3", "Vale revisa para baixo a projecao de producao de minerio para o ano"),
    ("2022-06-22", "BRAP4", "Bradespar acompanha valorizacao das acoes da Vale no trimestre"),
    ("2023-04-11", "BRAP4", "Bradespar reduz projecao de lucro apos resultado fraco da Vale"),
    ("2022-11-03", "ITUB4", "Itau Unibanco reporta lucro recorde impulsionado por credito consignado"),
    ("2023-07-14", "ITUB4", "Itau eleva provisao para devedores duvidosos em cenario de juros altos"),
    ("2022-09-27", "BBDC4", "Bradesco anuncia plano de reestruturacao para reduzir custos"),
    ("2023-03-08", "BBDC4", "Bradesco supera expectativas do mercado com alta na carteira de credito"),
    ("2022-07-19", "BBAS3", "Banco do Brasil registra crescimento robusto no agronegocio"),
    ("2023-05-30", "BBAS3", "Banco do Brasil eleva inadimplencia entre pequenas empresas"),
    ("2022-04-06", "MGLU3", "Magazine Luiza anuncia fechamento de lojas fisicas em reestruturacao"),
    ("2022-10-21", "MGLU3", "Magazine Luiza surpreende com crescimento das vendas online no trimestre"),
    ("2023-06-09", "MGLU3", "Acoes da Magazine Luiza caem apos resultado abaixo do esperado"),
    ("2022-12-02", "LREN3", "Lojas Renner reporta alta nas vendas de fim de ano"),
    ("2023-08-17", "LREN3", "Lojas Renner enfrenta aumento de custos com inadimplencia de clientes"),
    ("2022-02-14", "TOTS3", "Totvs expande receita recorrente com migracao de clientes para nuvem"),
    ("2023-10-25", "TOTS3", "Totvs anuncia aquisicao de startup de tecnologia financeira"),
    ("2022-08-30", "VIVT3", "Vivo anuncia expansao da cobertura 5G para novas cidades"),
    ("2023-02-06", "VIVT3", "Vivo mantem politica de dividendos estavel apesar de cenario de juros"),
    ("2023-11-13", "VIVT3", "Setor de telecomunicacoes enfrenta pressao regulatoria sobre tarifas")
]

df_noticias_pd = pd.DataFrame(manchetes, columns=["date", "ticker", "manchete"])
df_noticias_pd["nota"] = "manchete ficticia, criada para fins didaticos -- nao e noticia real"

df_noticias_pd.head()

## Passo 1 -- Salvando o Bronze

Mesmo padrao dos outros notebooks: dado bruto, sem transformacao, direto pra Delta Lake.

In [0]:
df_noticias_spark = spark.createDataFrame(df_noticias_pd)

df_noticias_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.bronze_noticias")

print("Bronze de noticias salvo com sucesso!")
print(f"Total de manchetes: {df_noticias_spark.count()}")

## Passo 2 -- Descobrindo qual LLM esta disponivel

O Databricks tem LLMs prontos pra usar direto na plataforma (Foundation Model API), sem precisar de conta ou chave de API externa. Mas o nome exato do endpoint disponivel pode variar. Rodo essa celula primeiro pra ver a lista e escolher um.

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

for endpoint in client.list_endpoints():
    print(endpoint["name"])

## Passo 3 -- Escolhendo o endpoint

Copia um dos nomes que apareceu na lista acima (geralmente algo comecando com `databricks-`, tipo `databricks-meta-llama-3-3-70b-instruct` ou `databricks-dbrx-instruct`) e cola abaixo.

In [0]:
NOME_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # troque pelo nome que apareceu na lista acima, se for diferente

## Passo 4 -- Classificando o sentimento com o LLM

Essa funcao manda a manchete pro modelo com uma instrucao clara (o **prompt**) pedindo uma resposta curta e padronizada -- so uma palavra, pra ficar facil de processar depois. Isso e diferente do XGBoost: aqui nao tem `.fit()`, o "aprendizado" ja veio pronto no modelo, eu so preciso pedir direito.

Faco isso num loop simples do Python, uma manchete de cada vez -- com so 24 linhas nao compensa a complexidade de paralelizar, e como cada chamada e uma requisicao de rede (nao processamento pesado), o Spark nao ajudaria aqui.

In [0]:
def classificar_sentimento(manchete):
    resposta = client.predict(
        endpoint=NOME_ENDPOINT,
        inputs={
            "messages": [
                {
                    "role": "system",
                    "content": "Voce classifica o sentimento de manchetes financeiras. "
                               "Responda com exatamente uma palavra: positivo, negativo ou neutro."
                },
                {"role": "user", "content": manchete}
            ],
            "max_tokens": 5,
            "temperature": 0
        }
    )
    texto = resposta["choices"][0]["message"]["content"].strip().lower()
    return texto

df_noticias_pd["sentimento"] = df_noticias_pd["manchete"].apply(classificar_sentimento)
df_noticias_pd[["ticker", "manchete", "sentimento"]]

## Passo 5 -- Transformando em numero e salvando o Silver

Pra virar feature de modelo, o sentimento em texto precisa virar numero: positivo = 1, neutro = 0, negativo = -1.

Um detalhe que so aparece na pratica: o LLM nem sempre responde a palavra exata igual todas as vezes -- as vezes vem `"positivo"`, as vezes `"positivo."` com ponto final. Se eu comparar por igualdade exata (`.map({"positivo": 1, ...})`), toda resposta com pontuacao a mais vira `null` sem avisar nada -- o notebook roda sem erro, mas o resultado fica errado silenciosamente, o que e pior que um erro visivel. Por isso uso `in` ("a palavra aparece dentro do texto?") em vez de igualdade exata -- funciona com ou sem pontuacao, maiuscula, espaco extra, etc.

In [0]:
def mapear_sentimento(texto):
    texto = texto.lower()
    if "positivo" in texto:
        return 1
    elif "negativo" in texto:
        return -1
    elif "neutro" in texto:
        return 0
    return None

df_noticias_pd["sentimento_score"] = df_noticias_pd["sentimento"].apply(mapear_sentimento)

df_silver_noticias = spark.createDataFrame(df_noticias_pd)

df_silver_noticias.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.silver_noticias")

print("Silver de noticias salvo com sucesso!")
display(df_silver_noticias)

## Passo 6 -- Agregando por ticker (Gold)

Com so 2-3 manchetes por acao, nao da pra casar por data exata com a `gold_ml_features` (a maioria dos dias nao teria manchete nenhuma). Entao, pra esse exercicio, agrego a **media de sentimento por ticker** -- uma visao geral de "esse papel teve mais noticia boa ou ruim no periodo", em vez de um sinal diario. E uma simplificacao proposital, coerente com o tamanho pequeno do dataset.

In [0]:
gold_sentimento = df_silver_noticias.groupBy("ticker") \
    .agg(
        F.avg("sentimento_score").alias("sentimento_medio"),
        F.count("*").alias("total_manchetes")
    ) \
    .orderBy(F.desc("sentimento_medio"))

gold_sentimento.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_sentimento_ticker")

print("Gold de sentimento salvo com sucesso!")
display(gold_sentimento)

## Conclusao

Esse notebook fecha a parte de GenAI do projeto: um LLM hospedado no Databricks classificou manchetes sem nenhum treino especifico da minha parte, so com uma instrucao bem escrita -- e o resultado virou uma tabela Gold, seguindo o mesmo padrao Bronze/Silver/Gold do resto do pipeline.

Como comentado no inicio, esse dataset e pequeno demais pra provar estatisticamente que sentimento melhora a previsao de retorno (isso exigiria noticias reais em volume bem maior). O valor aqui foi demonstrar a tecnica -- classificacao de texto com GenAI integrada ao mesmo pipeline de dados -- de forma honesta sobre o que da e o que nao da pra concluir com esse volume de dado.